In [1]:
# ============================================================
# Embedding Model Selection
# LaBSE vs multilingual-E5-large
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import random
import torch

PROJECT_ROOT = Path(
    "/home/jovyan/project work/data_analyssis"
)

TRAIN_PATH = (
    PROJECT_ROOT /
    "RUHSOLD_train.tsv"
)

SEED = 42
N_QUERIES_PER_CLASS = 20

print("Training file exists:", TRAIN_PATH.exists())

[HAMI-core Msg(1153:140486755200320:libvgpu.c:839)]: Initializing.....


Training file exists: True


In [2]:
# ============================================================
# Load authentic RUHSOLD training data
# ============================================================

train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t",
    names=["text", "label"]
)

ID_TO_LABEL = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}

train_df["class_name"] = (
    train_df["label"]
    .map(ID_TO_LABEL)
)

print("Training samples:", len(train_df))

print("\nClass distribution:")
display(
    train_df["class_name"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

Training samples: 6408

Class distribution:


,Class,Count
0,Normal,3423
1,Abusive/Offensive,1537
2,Sexism,537
3,Religious Hate,500
4,Profane,411


In [3]:
# ============================================================
# Balanced query subset
# 20 samples per RUHSOLD class
# ============================================================

query_df = (
    train_df
    .groupby(
        "class_name",
        group_keys=False
    )
    .sample(
        n=N_QUERIES_PER_CLASS,
        random_state=SEED
    )
    .copy()
)

# Preserve original training-row index.
query_df = (
    query_df
    .reset_index()
    .rename(
        columns={
            "index": "source_index"
        }
    )
)

print("Number of queries:", len(query_df))

print("\nQueries per class:")
display(
    query_df["class_name"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

display(query_df.head())

Number of queries: 100

Queries per class:


,Class,Count
0,Abusive/Offensive,20
1,Normal,20
2,Profane,20
3,Religious Hate,20
4,Sexism,20


,source_index,text,label,class_name
0,4877,i swearrrrr. matlab banda aram se europe mein ...,0,Abusive/Offensive
1,287,chutiye kaagaz nhi dikhane caa mein😂😂 charsi.,0,Abusive/Offensive
2,1290,lifafa ni mil raha is haram khor kanjre k bach...,0,Abusive/Offensive
3,4394,shown more balls than many phattu bollywood tw...,0,Abusive/Offensive
4,5662,ka jri ur rundi ka bacha tuje paida jerne waal...,0,Abusive/Offensive


In [4]:
# ============================================================
# Retrieval evaluation
# ============================================================

def evaluate_same_label_retrieval(
    neighbour_indices,
    query_source_indices,
    query_labels,
    corpus_labels,
    k
):
    """
    Calculate same-label Precision@k.

    Self-matches are excluded using original
    training-set row indices.
    """

    scores = []

    for i, neighbours in enumerate(neighbour_indices):

        query_index = query_source_indices[i]
        query_label = query_labels[i]

        valid_neighbours = []

        for neighbour_index in neighbours:

            # Exclude the query sentence itself.
            if neighbour_index == query_index:
                continue

            valid_neighbours.append(
                neighbour_index
            )

            if len(valid_neighbours) == k:
                break

        neighbour_labels = [
            corpus_labels[idx]
            for idx in valid_neighbours
        ]

        same_label_count = sum(
            label == query_label
            for label in neighbour_labels
        )

        scores.append(
            same_label_count / k
        )

    return np.array(scores)

In [5]:
# ============================================================
# LaBSE + FAISS setup
# ============================================================

%pip install -q sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [6]:
from sentence_transformers import SentenceTransformer
import faiss

print("Sentence-Transformers and FAISS imported successfully.")

Sentence-Transformers and FAISS imported successfully.


In [7]:
# ============================================================
# Load LaBSE
# ============================================================

LABSE_MODEL_NAME = "sentence-transformers/LaBSE"

labse_model = SentenceTransformer(
    LABSE_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Loaded:", LABSE_MODEL_NAME)
print("Device:", labse_model.device)

[HAMI-core Msg(1153:140486755200320:libvgpu.c:855)]: Initialized


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Loaded: sentence-transformers/LaBSE
Device: cuda:0


In [8]:
# ============================================================
# Encode RUHSOLD training corpus with LaBSE
# ============================================================

labse_corpus_embeddings = labse_model.encode(
    train_df["text"].astype(str).tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Corpus embedding shape:",
    labse_corpus_embeddings.shape
)

Batches:   0%|          | 0/101 [00:00<?, ?it/s]

Corpus embedding shape: (6408, 768)


In [9]:
# ============================================================
# Build exact FAISS index for LaBSE
# ============================================================

embedding_dim = (
    labse_corpus_embeddings.shape[1]
)

labse_index = faiss.IndexFlatIP(
    embedding_dim
)

labse_index.add(
    labse_corpus_embeddings.astype(
        np.float32
    )
)

print(
    "Vectors indexed:",
    labse_index.ntotal
)

Vectors indexed: 6408


In [10]:
# ============================================================
# Encode balanced query subset
# ============================================================

labse_query_embeddings = labse_model.encode(
    query_df["text"].astype(str).tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Query embedding shape:",
    labse_query_embeddings.shape
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Query embedding shape: (100, 768)


In [11]:
# ============================================================
# Retrieve nearest neighbours with LaBSE
# ============================================================

SEARCH_K = 11

labse_scores, labse_indices = (
    labse_index.search(
        labse_query_embeddings.astype(
            np.float32
        ),
        SEARCH_K
    )
)

print(
    "Retrieved neighbour matrix:",
    labse_indices.shape
)

Retrieved neighbour matrix: (100, 11)


In [12]:
# ============================================================
# LaBSE retrieval evaluation
# ============================================================

query_source_indices = (
    query_df["source_index"]
    .astype(int)
    .tolist()
)

query_labels = (
    query_df["label"]
    .astype(int)
    .tolist()
)

corpus_labels = (
    train_df["label"]
    .astype(int)
    .tolist()
)

labse_p5_scores = evaluate_same_label_retrieval(
    neighbour_indices=labse_indices,
    query_source_indices=query_source_indices,
    query_labels=query_labels,
    corpus_labels=corpus_labels,
    k=5
)

labse_p10_scores = evaluate_same_label_retrieval(
    neighbour_indices=labse_indices,
    query_source_indices=query_source_indices,
    query_labels=query_labels,
    corpus_labels=corpus_labels,
    k=10
)

LABSE_P5 = labse_p5_scores.mean()
LABSE_P10 = labse_p10_scores.mean()

print("LaBSE SAME-LABEL RETRIEVAL")
print("=" * 40)

print(
    f"Precision@5 : "
    f"{LABSE_P5:.4f}"
)

print(
    f"Precision@10: "
    f"{LABSE_P10:.4f}"
)

LaBSE SAME-LABEL RETRIEVAL
Precision@5 : 0.3740
Precision@10: 0.3670


In [13]:
# ============================================================
# Class-wise LaBSE retrieval performance
# ============================================================

labse_results_df = query_df[
    [
        "source_index",
        "text",
        "label",
        "class_name"
    ]
].copy()

labse_results_df["p_at_5"] = (
    labse_p5_scores
)

labse_results_df["p_at_10"] = (
    labse_p10_scores
)

labse_class_results = (
    labse_results_df
    .groupby("class_name")
    .agg(
        queries=("source_index", "count"),
        precision_at_5=("p_at_5", "mean"),
        precision_at_10=("p_at_10", "mean")
    )
    .reset_index()
)

display(
    labse_class_results.round(4)
)

,class_name,queries,precision_at_5,precision_at_10
0,Abusive/Offensive,20,0.46,0.465
1,Normal,20,0.72,0.690
2,Profane,20,0.19,0.220
3,Religious Hate,20,0.20,0.190
4,Sexism,20,0.30,0.270


In [16]:
# ============================================================
# Inspect LaBSE nearest-neighbour examples
# ============================================================

def show_neighbours(
    query_position,
    scores,
    indices,
    k=5
):
    query_row = query_df.iloc[
        query_position
    ]

    print("=" * 90)

    print("QUERY:")
    print(query_row["text"])

    print(
        "\nQUERY CLASS:",
        query_row["class_name"]
    )

    print("\nNEAREST NEIGHBOURS:\n")

    shown = 0

    for score, idx in zip(
        scores[query_position],
        indices[query_position]
    ):

        if idx == query_row["source_index"]:
            continue

        neighbour = train_df.iloc[idx]

        print(
            f"Similarity: {score:.4f}"
        )

        print(
            "Class:",
            neighbour["class_name"]
        )

        print(
            "Text:",
            neighbour["text"]
        )

        print("-" * 90)

        shown += 1

        if shown == k:
            break

In [15]:
for class_name in ID_TO_LABEL.values():

    position = (
        query_df[
            query_df["class_name"]
            == class_name
        ]
        .index[0]
    )

    show_neighbours(
        query_position=position,
        scores=labse_scores,
        indices=labse_indices,
        k=5
    )

QUERY:
i swearrrrr. matlab banda aram se europe mein rehta, aur yahan bhangiyon ke liye jaan maar raha.

QUERY CLASS: Abusive/Offensive

NEAREST NEIGHBOURS:

Similarity: 0.5521
Class: Normal
Text: raabi u r my favrit aur aap ko bht bht mubarak naye zindage shuru krny prr
------------------------------------------------------------------------------------------
Similarity: 0.5065
Class: Profane
Text:  tm jao ma ni ja rha, pehly he adhi zindagi saffr mein guzr gai bc meri.
------------------------------------------------------------------------------------------
Similarity: 0.4811
Class: Religious Hate
Text:  mgar shram tum ko aati ni, itna bughz muhammad o aal e muhammad say? kis moun k sath qbar mai jao ge.
------------------------------------------------------------------------------------------
Similarity: 0.4728
Class: Sexism
Text: apni ammi aur bahan ke sath aise hi baat karte hain isiliye tere country mein sab log hijra ha🤣🤣🤣🐐
------------------------------------------------------

In [17]:
# ============================================================
# Load multilingual-E5-large
# ============================================================

E5_MODEL_NAME = "intfloat/multilingual-e5-large"

e5_model = SentenceTransformer(
    E5_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Loaded:", E5_MODEL_NAME)
print("Device:", e5_model.device)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Loaded: intfloat/multilingual-e5-large
Device: cuda:0


In [18]:
# ============================================================
# Prepare E5 inputs
# ============================================================

e5_corpus_texts = [
    "query: " + str(text)
    for text in train_df["text"].tolist()
]

e5_query_texts = [
    "query: " + str(text)
    for text in query_df["text"].tolist()
]

print("Corpus texts:", len(e5_corpus_texts))
print("Query texts :", len(e5_query_texts))

print("\nExample:")
print(e5_query_texts[0])

Corpus texts: 6408
Query texts : 100

Example:
query: i swearrrrr. matlab banda aram se europe mein rehta, aur yahan bhangiyon ke liye jaan maar raha.


In [19]:
# ============================================================
# Encode RUHSOLD training corpus with E5
# ============================================================

e5_corpus_embeddings = e5_model.encode(
    e5_corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Corpus embedding shape:",
    e5_corpus_embeddings.shape
)

Batches:   0%|          | 0/201 [00:00<?, ?it/s]

Corpus embedding shape: (6408, 1024)


In [20]:
# ============================================================
# Build exact FAISS index for E5
# ============================================================

e5_embedding_dim = (
    e5_corpus_embeddings.shape[1]
)

e5_index = faiss.IndexFlatIP(
    e5_embedding_dim
)

e5_index.add(
    e5_corpus_embeddings.astype(
        np.float32
    )
)

print(
    "Vectors indexed:",
    e5_index.ntotal
)

print(
    "Embedding dimension:",
    e5_embedding_dim
)

Vectors indexed: 6408
Embedding dimension: 1024


In [21]:
# ============================================================
# Encode balanced query subset with E5
# ============================================================

e5_query_embeddings = e5_model.encode(
    e5_query_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Query embedding shape:",
    e5_query_embeddings.shape
)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Query embedding shape: (100, 1024)


In [22]:
# ============================================================
# Retrieve nearest neighbours with E5
# ============================================================

SEARCH_K = 11

e5_scores, e5_indices = (
    e5_index.search(
        e5_query_embeddings.astype(
            np.float32
        ),
        SEARCH_K
    )
)

print(
    "Retrieved neighbour matrix:",
    e5_indices.shape
)

Retrieved neighbour matrix: (100, 11)


In [23]:
# ============================================================
# E5 retrieval evaluation
# ============================================================

e5_p5_scores = evaluate_same_label_retrieval(
    neighbour_indices=e5_indices,
    query_source_indices=query_source_indices,
    query_labels=query_labels,
    corpus_labels=corpus_labels,
    k=5
)

e5_p10_scores = evaluate_same_label_retrieval(
    neighbour_indices=e5_indices,
    query_source_indices=query_source_indices,
    query_labels=query_labels,
    corpus_labels=corpus_labels,
    k=10
)

E5_P5 = e5_p5_scores.mean()
E5_P10 = e5_p10_scores.mean()

print("MULTILINGUAL-E5-LARGE SAME-LABEL RETRIEVAL")
print("=" * 50)

print(
    f"Precision@5 : "
    f"{E5_P5:.4f}"
)

print(
    f"Precision@10: "
    f"{E5_P10:.4f}"
)

MULTILINGUAL-E5-LARGE SAME-LABEL RETRIEVAL
Precision@5 : 0.4360
Precision@10: 0.4050


In [24]:
# ============================================================
# Class-wise E5 retrieval performance
# ============================================================

e5_results_df = query_df[
    [
        "source_index",
        "text",
        "label",
        "class_name"
    ]
].copy()

e5_results_df["p_at_5"] = (
    e5_p5_scores
)

e5_results_df["p_at_10"] = (
    e5_p10_scores
)

e5_class_results = (
    e5_results_df
    .groupby("class_name")
    .agg(
        queries=("source_index", "count"),
        precision_at_5=("p_at_5", "mean"),
        precision_at_10=("p_at_10", "mean")
    )
    .reset_index()
)

display(
    e5_class_results.round(4)
)

,class_name,queries,precision_at_5,precision_at_10
0,Abusive/Offensive,20,0.38,0.385
1,Normal,20,0.72,0.700
2,Profane,20,0.29,0.250
3,Religious Hate,20,0.34,0.305
4,Sexism,20,0.45,0.385


In [25]:
# ============================================================
# LaBSE vs multilingual-E5-large
# ============================================================

overall_comparison = pd.DataFrame({
    "model": [
        "LaBSE",
        "multilingual-E5-large"
    ],
    "precision_at_5": [
        LABSE_P5,
        E5_P5
    ],
    "precision_at_10": [
        LABSE_P10,
        E5_P10
    ]
})

display(
    overall_comparison.round(4)
)

,model,precision_at_5,precision_at_10
0,LaBSE,0.374,0.367
1,multilingual-E5-large,0.436,0.405


In [26]:
# ============================================================
# Class-wise model comparison
# ============================================================

labse_compare = (
    labse_class_results
    .rename(
        columns={
            "precision_at_5": "labse_p5",
            "precision_at_10": "labse_p10"
        }
    )
)

e5_compare = (
    e5_class_results[
        [
            "class_name",
            "precision_at_5",
            "precision_at_10"
        ]
    ]
    .rename(
        columns={
            "precision_at_5": "e5_p5",
            "precision_at_10": "e5_p10"
        }
    )
)

class_comparison = (
    labse_compare
    .merge(
        e5_compare,
        on="class_name",
        how="inner"
    )
)

display(
    class_comparison.round(4)
)

,class_name,queries,labse_p5,labse_p10,e5_p5,e5_p10
0,Abusive/Offensive,20,0.46,0.465,0.38,0.385
1,Normal,20,0.72,0.690,0.72,0.700
2,Profane,20,0.19,0.220,0.29,0.250
3,Religious Hate,20,0.20,0.190,0.34,0.305
4,Sexism,20,0.30,0.270,0.45,0.385
